# Visualize Alignment Optimization Results

This notebook demonstrates how to load and visualize the optimization data captured during tilt-series alignment.

**Use this for:**
- Creating presentation-quality figures
- Exploring optimization behavior
- Analyzing precision evolution
- Comparing subvolumes across optimization steps

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

from miss_alignment.alignment.visualize_alignment import load_optimization_data

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Optimization Data

In [ ]:
# Set path to your optimization tracking directory
tracking_dir = Path("/path/to/output/optimization_steps")

# Load all step data
step_data = load_optimization_data(tracking_dir, load_subvolumes=True)

print(f"Loaded {len(step_data)} optimization steps")
print(f"Initial loss: {step_data[0].loss:.6f}")
print(f"Final loss: {step_data[-1].loss:.6f}")
print(f"Loss reduction: {step_data[0].loss - step_data[-1].loss:.6f}")

## 2. Plot Loss and Precision Evolution

In [ ]:
steps = [s.step for s in step_data]
losses = [s.loss for s in step_data]
mean_precisions = [s.mean_precision for s in step_data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(steps, losses, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Optimization Step', fontsize=14)
ax1.set_ylabel('Loss (Precision-Weighted Score)', fontsize=14)
ax1.set_title('Loss Evolution', fontsize=16, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Precision curve
ax2.plot(steps, mean_precisions, 's-', linewidth=2, markersize=8, color='C1')
ax2.set_xlabel('Optimization Step', fontsize=14)
ax2.set_ylabel('Mean Precision', fontsize=14)
ax2.set_title('Precision Evolution', fontsize=16, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Visualize Subvolumes at Different Steps

Compare reconstruction quality at different optimization steps.

In [ ]:
# Select steps to visualize (first, middle, last)
step_indices = [0, len(step_data) // 2, -1]
subvolume_idx = 0  # Which subvolume to display
z_slice = 48  # Which Z slice to show (middle of 96-pixel volume)

fig, axes = plt.subplots(1, len(step_indices), figsize=(15, 5))

for i, step_idx in enumerate(step_indices):
    data = step_data[step_idx]
    
    if data.subvolumes is not None and data.subvolumes.shape[0] > subvolume_idx:
        subvol = data.subvolumes[subvolume_idx, z_slice, :, :].numpy()
        precision = data.precisions[subvolume_idx].item() if data.precisions is not None else 0
        
        axes[i].imshow(subvol, cmap='gray')
        axes[i].set_title(
            f'Step {data.step}\nLoss: {data.loss:.4f}\nPrecision: {precision:.4f}',
            fontsize=12
        )
        axes[i].axis('off')
    else:
        axes[i].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[i].transAxes)
        axes[i].axis('off')

plt.suptitle(f'Subvolume Evolution (Index {subvolume_idx}, Z-slice {z_slice})', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Analyze Precision Distribution

Visualize how model precision changes across the volume during optimization.

In [ ]:
# Get precision distributions at different steps
step_indices = [0, len(step_data) // 2, -1]

fig, axes = plt.subplots(1, len(step_indices), figsize=(15, 4))

for i, step_idx in enumerate(step_indices):
    data = step_data[step_idx]
    
    if data.precisions is not None:
        precisions = data.precisions.numpy()
        
        axes[i].hist(precisions, bins=30, alpha=0.7, edgecolor='black')
        axes[i].axvline(precisions.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {precisions.mean():.3f}')
        axes[i].set_xlabel('Precision', fontsize=12)
        axes[i].set_ylabel('Count', fontsize=12)
        axes[i].set_title(f'Step {data.step}', fontsize=12, fontweight='bold')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.suptitle('Precision Distribution Evolution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Shift Evolution Analysis

Track how alignment shifts change during optimization.

In [ ]:
if step_data[0].shifts_x is not None:
    # Extract shifts
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)  # (n_steps, n_tilts)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    
    # Compute shift magnitude
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot mean and individual tilts
    mean_magnitude = shift_magnitude.mean(dim=1)
    ax.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean', zorder=10)
    
    for i in range(shifts_x.shape[1]):
        ax.plot(steps, shift_magnitude[:, i], alpha=0.2, color='C0', linewidth=1)
    
    ax.set_xlabel('Optimization Step', fontsize=14)
    ax.set_ylabel('Shift Magnitude (Angstroms)', fontsize=14)
    ax.set_title('Shift Evolution During Optimization', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No shift data available")

## 6. Create Publication-Quality Figure

Combine multiple visualizations into a single comprehensive figure.

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Loss curve (top left)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, losses, 'o-', linewidth=2, markersize=6)
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('A. Loss Evolution', fontsize=14, fontweight='bold', loc='left')
ax1.grid(True, alpha=0.3)

# Precision curve (top middle)
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(steps, mean_precisions, 's-', linewidth=2, markersize=6, color='C1')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Mean Precision', fontsize=12)
ax2.set_title('B. Precision Evolution', fontsize=14, fontweight='bold', loc='left')
ax2.grid(True, alpha=0.3)

# Precision distribution (top right)
ax3 = fig.add_subplot(gs[0, 2])
if step_data[-1].precisions is not None:
    final_precisions = step_data[-1].precisions.numpy()
    ax3.hist(final_precisions, bins=30, alpha=0.7, edgecolor='black')
    ax3.axvline(final_precisions.mean(), color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('Precision', fontsize=12)
ax3.set_ylabel('Count', fontsize=12)
ax3.set_title('C. Final Precision Distribution', fontsize=14, fontweight='bold', loc='left')
ax3.grid(True, alpha=0.3)

# Subvolume comparison (middle row)
subvolume_idx = 0
z_slice = 48
step_indices = [0, len(step_data) // 2, -1]
labels = ['D. Initial', 'E. Middle', 'F. Final']

for i, (step_idx, label) in enumerate(zip(step_indices, labels)):
    ax = fig.add_subplot(gs[1, i])
    data = step_data[step_idx]
    
    if data.subvolumes is not None and data.subvolumes.shape[0] > subvolume_idx:
        subvol = data.subvolumes[subvolume_idx, z_slice, :, :].numpy()
        im = ax.imshow(subvol, cmap='gray')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f'{label}\nStep {data.step}, Loss: {data.loss:.4f}', fontsize=12, fontweight='bold', loc='left')
    ax.axis('off')

# Shift evolution (bottom row, spanning all columns)
ax7 = fig.add_subplot(gs[2, :])
if step_data[0].shifts_x is not None:
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    mean_magnitude = shift_magnitude.mean(dim=1)
    
    ax7.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean shift magnitude', zorder=10)
    for i in range(shifts_x.shape[1]):
        ax7.plot(steps, shift_magnitude[:, i], alpha=0.15, color='C0', linewidth=1)
    
    ax7.set_xlabel('Step', fontsize=12)
    ax7.set_ylabel('Shift Magnitude (Å)', fontsize=12)
    ax7.set_title('G. Shift Evolution', fontsize=14, fontweight='bold', loc='left')
    ax7.legend(fontsize=10)
    ax7.grid(True, alpha=0.3)

plt.suptitle('Alignment Optimization Progress', fontsize=18, fontweight='bold', y=0.995)
plt.savefig('optimization_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved as 'optimization_summary.png'")

## 7. Export Data for Further Analysis

In [ ]:
# Export summary statistics to CSV
import pandas as pd

summary_df = pd.DataFrame({
    'step': steps,
    'loss': losses,
    'mean_precision': mean_precisions,
})

summary_df.to_csv('optimization_summary.csv', index=False)
print("Summary statistics saved to 'optimization_summary.csv'")

# Display summary
summary_df

In [ ]:
# Alternative: Generate all 3D visualizations at once
from miss_alignment.alignment.visualize_alignment import create_3d_visualizations

# Uncomment to run:
# create_3d_visualizations(
#     tracking_dir=tracking_dir,
#     output_dir=Path("3d_visualizations"),
#     create_gifs=True,
#     gif_duration=500,
#     elev=30,
#     azim=45,
# )

print("To generate all 3D visualizations at once, uncomment and run the cell above.")

### 8.4 Alternative: Use the Convenience Function

Instead of manually creating plots, you can use the convenience function that generates everything at once.

In [ ]:
from IPython.display import Image as IPImage, display

print("Precision Evolution Animation:")
display(IPImage(filename="precision_evolution.gif"))

print("\nLoss Evolution Animation:")
display(IPImage(filename="loss_evolution.gif"))

### 8.3 Display Generated GIFs

Display the generated animated GIFs in the notebook.

In [ ]:
from miss_alignment.alignment.visualize_alignment import (
    generate_3d_animation_frames,
    save_animation_as_gif,
)

# Generate precision animation
print("Generating precision animation frames...")
precision_frames = generate_3d_animation_frames(
    step_data,
    value_key="precision",
    cmap="viridis",
    elev=30,
    azim=45,
)

print(f"Saving precision animation ({len(precision_frames)} frames)...")
save_animation_as_gif(
    precision_frames,
    Path("precision_evolution.gif"),
    duration=500,  # milliseconds per frame
    dpi=100,
)
print("✓ Saved: precision_evolution.gif")

# Generate loss animation
print("\nGenerating loss animation frames...")
loss_frames = generate_3d_animation_frames(
    step_data,
    value_key="loss",
    cmap="coolwarm",
    elev=30,
    azim=45,
)

print(f"Saving loss animation ({len(loss_frames)} frames)...")
save_animation_as_gif(
    loss_frames,
    Path("loss_evolution.gif"),
    duration=500,
    dpi=100,
)
print("✓ Saved: loss_evolution.gif")

### 8.2 Generate Animated GIFs

Create animated GIFs showing how precision and loss evolve across optimization steps.

**Note:** This can take several minutes depending on the number of steps.

In [ ]:
# Create loss plots
fig = create_3d_scatter_plot(
    positions=initial_step.positions,
    values=initial_step.scores,
    title=f"Initial Loss Distribution (Step {initial_step.step})",
    colorbar_label="Loss",
    cmap="coolwarm",
    elev=30,
    azim=45,
)
plt.show()

fig = create_3d_scatter_plot(
    positions=final_step.positions,
    values=final_step.scores,
    title=f"Final Loss Distribution (Step {final_step.step})",
    colorbar_label="Loss",
    cmap="coolwarm",
    elev=30,
    azim=45,
)
plt.show()

In [ ]:
from miss_alignment.alignment.visualize_alignment import create_3d_scatter_plot

# Select initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

# Create precision plots
fig = create_3d_scatter_plot(
    positions=initial_step.positions,
    values=initial_step.precisions,
    title=f"Initial Precision Distribution (Step {initial_step.step})",
    colorbar_label="Precision",
    cmap="viridis",
    elev=30,
    azim=45,
)
plt.show()

fig = create_3d_scatter_plot(
    positions=final_step.positions,
    values=final_step.precisions,
    title=f"Final Precision Distribution (Step {final_step.step})",
    colorbar_label="Precision",
    cmap="viridis",
    elev=30,
    azim=45,
)
plt.show()

### 8.1 Static 3D Scatter Plots

Create 3D scatter plots showing subvolume positions colored by precision or loss.

In [ ]:
# Position data is always available in modern optimization runs
print(f"Found {len(step_data)} optimization steps with position data")

## 8. 3D Visualization of Subvolume Positions

Visualize the spatial distribution of subvolumes in 3D and how precision/loss varies across the volume.

In [ ]:
from IPython.display import Image as IPImage, display, HTML
from pathlib import Path

# List of generated GIF files
projection_names = ['depth', 'height', 'width']
gif_types = ['precision_weighted_grid', 'loss_overlay_grid']

print("Generated Animations:\n")

for gif_type in gif_types:
    print(f"\n{'='*60}")
    print(f"{gif_type.replace('_', ' ').title()}")
    print('='*60)
    
    for proj_name in projection_names:
        gif_path = f'{gif_type}_{proj_name}_evolution.gif'
        
        if Path(gif_path).exists():
            print(f"\n{proj_name.capitalize()} Projection:")
            display(IPImage(filename=gif_path))
        else:
            print(f"\n{proj_name.capitalize()} Projection: File not found ({gif_path})")

### 9.5 Display Generated Animations

Display the generated GIF animations in the notebook.

In [ ]:
import io
from PIL import Image

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate animations for each projection direction
for proj_dim in range(3):
    print(f"Generating loss overlay animation for {projection_names[proj_dim]} projection...")
    
    frames = []
    
    for step_data_item in step_data:
        # Create grid for this step
        grid, scores_grid = create_loss_overlay_grid(
            step_data_item.subvolumes,
            step_data_item.scores,
            projection_dim=proj_dim,
            normalize_scores=True
        )
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 10))
        
        # Show grayscale background with loss overlay
        ax.imshow(grid, cmap='gray', interpolation='bilinear', alpha=0.6)
        im = ax.imshow(scores_grid, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
        
        ax.set_title(
            f'{projection_names[proj_dim]} Projection - Step {step_data_item.step}\n'
            f'Loss: {step_data_item.loss:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Normalized Loss', rotation=270, labelpad=20)
        
        plt.tight_layout()
        
        # Render to image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)
    
    # Save as GIF
    if frames:
        output_path = f'loss_overlay_grid_{projection_names[proj_dim].split()[0].lower()}_evolution.gif'
        frames[0].save(
            output_path,
            save_all=True,
            append_images=frames[1:],
            duration=500,  # 500ms per frame
            loop=0,
            optimize=False,
        )
        print(f"✓ Saved: {output_path} ({len(frames)} frames)")
    else:
        print(f"  No frames generated for {projection_names[proj_dim]} projection")

print("\n✓ All loss overlay animations complete!")

### 9.4 Animated Loss Overlay Grid

Create animated GIFs showing how loss values across subvolumes evolve during optimization.

In [ ]:
import io
from PIL import Image

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate animations for each projection direction
for proj_dim in range(3):
    print(f"Generating precision-weighted animation for {projection_names[proj_dim]} projection...")
    
    frames = []
    
    for step_data_item in step_data:
        # Create grid for this step
        grid = create_precision_weighted_grid(
            step_data_item.subvolumes,
            step_data_item.precisions,
            projection_dim=proj_dim,
            normalize_precision=True
        )
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 10))
        im = ax.imshow(grid, cmap='gray', interpolation='bilinear')
        ax.set_title(
            f'{projection_names[proj_dim]} Projection - Step {step_data_item.step}\n'
            f'Loss: {step_data_item.loss:.4f}, Mean Precision: {step_data_item.mean_precision:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Precision-Weighted Intensity', rotation=270, labelpad=20)
        
        plt.tight_layout()
        
        # Render to image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)
    
    # Save as GIF
    if frames:
        output_path = f'precision_weighted_grid_{projection_names[proj_dim].split()[0].lower()}_evolution.gif'
        frames[0].save(
            output_path,
            save_all=True,
            append_images=frames[1:],
            duration=500,  # 500ms per frame
            loop=0,
            optimize=False,
        )
        print(f"✓ Saved: {output_path} ({len(frames)} frames)")
    else:
        print(f"  No frames generated for {projection_names[proj_dim]} projection")

print("\n✓ All precision-weighted animations complete!")

### 9.3 Animated Precision-Weighted Grid

Create animated GIFs showing how precision-weighted subvolume projections evolve during optimization.

In [ ]:
# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial, scores_grid_initial = create_loss_overlay_grid(
        initial_step.subvolumes,
        initial_step.scores,
        projection_dim=proj_dim,
        normalize_scores=True
    )
    
    # Show grayscale projection with loss overlay
    axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear', alpha=0.6)
    im0 = axes[0, proj_dim].imshow(scores_grid_initial, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')
    cbar0 = plt.colorbar(im0, ax=axes[0, proj_dim], fraction=0.046, pad=0.04)
    cbar0.set_label('Normalized Loss', rotation=270, labelpad=20)
    
    # Final step
    grid_final, scores_grid_final = create_loss_overlay_grid(
        final_step.subvolumes,
        final_step.scores,
        projection_dim=proj_dim,
        normalize_scores=True
    )
    
    axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear', alpha=0.6)
    im1 = axes[1, proj_dim].imshow(scores_grid_final, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')
    cbar1 = plt.colorbar(im1, ax=axes[1, proj_dim], fraction=0.046, pad=0.04)
    cbar1.set_label('Normalized Loss', rotation=270, labelpad=20)

plt.suptitle('Loss Overlay on Subvolume Grid', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_overlay_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: loss_overlay_grid_comparison.png")

### 9.2 Loss Overlay Grid Visualization

Show subvolume projections with loss values overlaid as a color map.

In [ ]:
# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial = create_precision_weighted_grid(
        initial_step.subvolumes,
        initial_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )
    
    im0 = axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear')
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}, Mean Precision: {initial_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')
    plt.colorbar(im0, ax=axes[0, proj_dim], fraction=0.046, pad=0.04)
    
    # Final step
    grid_final = create_precision_weighted_grid(
        final_step.subvolumes,
        final_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )
    
    im1 = axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear')
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}, Mean Precision: {final_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')
    plt.colorbar(im1, ax=axes[1, proj_dim], fraction=0.046, pad=0.04)

plt.suptitle('Precision-Weighted Subvolume Grid Visualization', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('precision_weighted_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: precision_weighted_grid_comparison.png")

### 9.1 Static Precision-Weighted Grid Visualization

Show precision-weighted projections for the first and last optimization steps.

In [ ]:
def create_precision_weighted_grid(subvolumes, precisions, projection_dim, normalize_precision=True):
    """Create a grid of precision-weighted subvolume projections.
    
    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    precisions : torch.Tensor
        Precision values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    normalize_precision : bool
        Whether to normalize precisions to [0, 1] range.
    
    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)  # +1 because first dim is batch
    
    # Normalize precisions to [0, 1]
    if normalize_precision:
        precision_min = precisions.min()
        precision_max = precisions.max()
        if precision_max > precision_min:
            precisions_norm = (precisions - precision_min) / (precision_max - precision_min)
        else:
            precisions_norm = torch.ones_like(precisions)
    else:
        precisions_norm = precisions
    
    # Apply precision weighting
    weighted_projections = projections * precisions_norm.view(-1, 1, 1)
    
    # Calculate grid dimensions
    n_patches = weighted_projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))
    
    # Get patch size
    patch_h, patch_w = weighted_projections.shape[1], weighted_projections.shape[2]
    
    # Create grid
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))
    
    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols
        
        patch = weighted_projections[idx].numpy()
        grid[row * patch_h:(row + 1) * patch_h, 
             col * patch_w:(col + 1) * patch_w] = patch
    
    return grid


def create_loss_overlay_grid(subvolumes, scores, projection_dim, normalize_scores=True):
    """Create a grid of subvolume projections colored by loss values.
    
    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    scores : torch.Tensor
        Score/loss values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    normalize_scores : bool
        Whether to normalize scores for visualization.
    
    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    scores_grid : np.ndarray
        Grid of score values for color overlay.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)
    
    # Normalize scores if requested
    if normalize_scores:
        score_min = scores.min()
        score_max = scores.max()
        if score_max > score_min:
            scores_norm = (scores - score_min) / (score_max - score_min)
        else:
            scores_norm = torch.ones_like(scores)
    else:
        scores_norm = scores
    
    # Calculate grid dimensions
    n_patches = projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))
    
    # Get patch size
    patch_h, patch_w = projections.shape[1], projections.shape[2]
    
    # Create grids
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))
    scores_grid = np.full((grid_rows * patch_h, grid_cols * patch_w), np.nan)
    
    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols
        
        patch = projections[idx].numpy()
        score_val = scores_norm[idx].item()
        
        grid[row * patch_h:(row + 1) * patch_h, 
             col * patch_w:(col + 1) * patch_w] = patch
        scores_grid[row * patch_h:(row + 1) * patch_h, 
                   col * patch_w:(col + 1) * patch_w] = score_val
    
    return grid, scores_grid

## 9. Precision-Weighted Subvolume Grid Visualization

Visualize all subvolumes in a grid, weighted by their precision values to show which patches the model is confident about.